<a href="https://colab.research.google.com/github/ash-myth/clinical-nlp-basics/blob/main/nlp_clinical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
text = "Patient has diabetes. She was prescribed metformin 500mg."
words = word_tokenize(text)
sentences = sent_tokenize(text)

In [ ]:
print("Words:", words)
print("Sentences:", sentences)

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
stopwords=set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean(text):
  text=text.lower()
  text=re.sub(r'[^a-z0-9\s]','',text)
  tokens=word_tokenize(text)
  tokens=[t for t in tokens if t not in stopwords]
  tokens=[lemmatizer.lemmatize(t) for t in tokens]
  return tokens

In [ ]:
print(clean("Patient was prescribed Metformin 500mg for Type-2 Diabetes!"))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
docs = [
    """Patient is a 56 year old male with a history of type 2 diabetes mellitus,
    hypertension, and obesity. Current medications include metformin 500mg twice daily
    and amlodipine 5mg once daily. Blood glucose remains elevated despite treatment.""",

    """A 42 year old female presented with persistent cough, fever, and shortness
    of breath. Chest X ray showed right lower lobe pneumonia. The patient was started
    on azithromycin and supportive care with close monitoring of oxygen saturation.""",

    """The patient reports severe chest pain radiating to the left arm along with
    sweating and nausea. ECG findings suggest acute myocardial infarction.
    Aspirin and nitroglycerin were administered immediately before transfer to ICU.""",

    """A 65 year old male with chronic kidney disease stage 3 attended follow up.
    Serum creatinine levels have increased over the past six months. Dietary sodium
    restriction and blood pressure control were emphasized during consultation.""",

    """A 29 year old female was diagnosed with iron deficiency anemia after
    laboratory investigations revealed low hemoglobin and ferritin levels.
    Oral iron supplementation was prescribed along with nutritional counseling."""
]

In [ ]:
vec=TfidfVectorizer(stop_words="english")
X=vec.fit_transform(docs)

In [ ]:
print("Features: ",vec.get_feature_names_out())
print("TF-IDF matrix shape: ",X.shape)

In [ ]:
import pandas as pd
df=pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out()
)
print(df.round(3))

In [ ]:
features=vec.get_feature_names_out()
for i,row in enumerate(X.toarray()):
    top_idx = row.argsort()[-5:][::-1]
    print(f"\nDocument {i+1}")
    for idx in top_idx:
        print(f"{features[idx]}: {row[idx]:.3f}")

In [ ]:
!pip install gensim numpy

In [ ]:
from gensim.models import Word2Vec

In [ ]:
raw_docs=[
    "Patient diagnosed with diabetes and hypertension",
    "Prescribed metformin for diabetes medication",
    "Hypertension treated with lisinopril medication",
    "Blood pressure is high due to hypertension",
    "Glucose levels are elevated because of diabetes",
    "Doctor prescribed medication to patient"
]
sentences=[clean(doc) for doc in raw_docs]

In [ ]:
sentences = [
    ["patient","diagnosed","diabetes","hypertension"],
    ["patient","suffers","diabetes"],
    ["metformin","used","treat","diabetes"],
    ["glucose","level","high","diabetes"],
    ["insulin","prescribed","diabetes"],
    ["blood","sugar","elevated","diabetes"],
    ["hypertension","treated","lisinopril"],
    ["blood","pressure","high","hypertension"],
    ["pressure","controlled","medication"],
    ["doctor","prescribed","metformin","patient"],
    ["glucose","monitoring","important","diabetes"],
    ["patient","takes","insulin","daily"]
]

In [ ]:
model = Word2Vec(
    sentences,
    vector_size=10,
    window=2,
    min_count=1,
    epochs=1000
)

In [ ]:
print("Vector for 'diabetes':", model.wv['diabetes'][:5], "...")
print("Words similar to 'diabetes':", model.wv.most_similar('diabetes'))

In [ ]:
print(model.wv.similarity("diabetes","glucose"))
print(model.wv.similarity("diabetes","lisinopril"))

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

In [ ]:
text = "Patient was diagnosed with Type 2 diabetes and prescribed metformin."
inputs=tokenizer(text,return_tensors="pt")

In [ ]:
print("Token IDs: ",inputs["input_ids"])
print("Tokens: ",tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

In [ ]:
with torch.no_grad():
  outputs=model(**inputs)

In [ ]:
last_hidden=outputs.last_hidden_state
cls_vec=last_hidden[0][0]

In [ ]:
print("Output shape:", last_hidden.shape)
print("[CLS] vector shape:", cls_vec.shape)
print("First 5 values of [CLS]:", cls_vec[:5])

In [ ]:
from transformers import pipeline

In [ ]:
ner=pipeline("ner",model="d4data/biomedical-ner-all",aggregation_strategy="simple")

In [ ]:
#text = "Patient with hypertension was prescribed lisinopril 10mg daily."
text="She has asthma and takes albuterol inhaler"
results = ner(text)

In [ ]:
for r in results:
    print(f"  {r['word']:<20} → {r['entity_group']:<15} (score: {r['score']:.2f})")

In [ ]:
!pip install spacy && python -m spacy download en_core_web_sm

In [ ]:
import spacy
from transformers import pipeline

In [ ]:
nlp=spacy.load("en_core_web_sm")
bio_ner=pipeline("ner",model="d4data/biomedical-ner-all",aggregation_strategy="simple")

In [ ]:
sentences = [
    "Patient with Type 2 diabetes was prescribed metformin 500mg twice daily.",
    "She has a history of hypertension and is on lisinopril 10mg.",
    "Chest pain and shortness of breath noted. ECG showed normal sinus rhythm."
]

In [ ]:
for sent in sentences:
  print(f"Sentence: {sent}")
  doc=nlp(sent)
  spacy_ents=[(e.text,e.label) for e in doc.ents]
  print("Entities: ", spacy_ents)
  bio_ents=bio_ner(sent)
  hf_ents=[(e["word"],e["entity_group"],round(e["score"],2))for e in bio_ents]
  print("BioBERT NER: ",hf_ents)

In [ ]:
def extract(text):
  ent={"drugs":[],"diseases":[],"other":[]}
  results=bio_ner(text)
  for r in results:
    label=r["entity_group"]
    if label=="Medication":
      ent["drugs"].append(r["word"])
    elif label=="Disease_disorder":
      ent["diseases"].append(r["word"])
    else:
      ent["other"].append((r["word"],label))
  return ent

In [ ]:
print("Structured extraction:")
print(extract("Patient with diabetes prescribed metformin 500mg twice daily."))

In [ ]:
results=bio_ner("Patient with Type 2 diabetes was prescribed metformin 500mg twice daily.")
for r in results:
    print(r['word'], '->', r['entity_group'], f"({r['score']:.2f})")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [ ]:
texts = [
    "Patient reports severe nausea and vomiting after medication",
    "Adverse reaction noted rash developed two days after dose",
    "Patient experienced dizziness and headache after drug administration",
    "Severe allergic reaction to penicillin anaphylaxis suspected",
    "Vomiting and diarrhea reported after starting new medication",
    "Patient developed skin rash and itching after treatment",
    "Shortness of breath observed after receiving medication",
    "Swelling of lips and tongue after antibiotic use",
    "Patient complained of severe chest pain after taking medication",
    "Drug induced liver injury suspected due to elevated enzymes",
    "Patient developed hives after vaccine administration",
    "Blurred vision and dizziness reported after therapy",
    "Medication caused persistent fatigue and weakness",
    "Patient experienced severe abdominal pain after treatment",
    "Allergic response observed after administration of contrast dye",
    "Nausea persisted despite discontinuation of medication",
    "Patient reported rapid heartbeat after taking drug",
    "Severe migraine developed after new prescription",
    "Fever and rash appeared after antibiotic therapy",
    "Patient experienced confusion and drowsiness after medication",
    "Blood pressure readings normal patient stable",
    "No complaints routine checkup all vitals normal",
    "Lab results within normal range no issues",
    "Patient doing well follow up in three months",
    "Patient tolerating treatment well with no side effects",
    "Routine diabetes follow up blood sugar controlled",
    "Patient reports feeling healthy and energetic",
    "Medication effective with no adverse symptoms",
    "Physical examination unremarkable patient stable",
    "Normal ECG findings no abnormalities detected",
    "Patient recovering well after surgery",
    "Kidney function tests within normal limits",
    "Patient reports improved symptoms after treatment",
    "Routine vaccination completed without complications",
    "Normal liver enzyme levels observed",
    "Patient sleeping well and maintaining appetite",
    "Follow up examination showed steady improvement",
    "No evidence of infection patient healthy",
    "Blood glucose levels well controlled",
    "Patient continues medication without problems",
    "Regular monitoring shows stable condition",
    "No allergic reactions reported during therapy",
    "Patient remains symptom free after treatment",
    "Vital signs normal throughout hospital stay",
    "Therapy completed successfully without complications",
    "Patient reports increased mobility and comfort",
    "Chest X ray normal with no findings",
    "Routine health screening results normal",
    "Patient discharged in stable condition",
    "No significant complaints during consultation"
]

labels = [
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0
]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(texts,labels,test_size=0.2,random_state=42)

In [ ]:
vec=TfidfVectorizer(ngram_range=(1,2))
xtr=vec.fit_transform(X_train)
xte=vec.transform(X_test)

In [ ]:
clf=LogisticRegression()
clf.fit(xtr,y_train)
pred=clf.predict(xte)

In [ ]:
print(classification_report(y_test,pred,target_names=["Normal","Adverse Event"]))

In [ ]:
notes=[
    "Patient developed severe rash after taking amoxicillin",
    "Blood pressure within normal limits and patient stable",
    "Vomiting and nausea started after medication use",
    "Routine follow up visit patient doing well",
    "Anaphylactic reaction observed following penicillin administration",
    "No side effects reported during treatment",
    "Patient complained of dizziness and blurred vision after therapy",
    "Laboratory values normal and patient healthy",
    "Severe allergic reaction with swelling of lips and tongue",
    "Patient discharged in stable condition"
]
for note in notes:
  nvec=vec.transform([note])
  pred=clf.predict(nvec)[0]
  prob=clf.predict_proba(nvec)[0]
  print(f"Prediction for {note}: ", "Adverse Event" if pred else "Normal")
  print(f"Confidence: {max(prob):.0%}")

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2"
)
model=AutoModel.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2"
)

In [ ]:
def get_embedding(text):
    inputs=tokenizer(text,return_tensors="pt",truncation=True,padding=True,max_length=128)
    with torch.no_grad():
        outputs=model(**inputs)
    return outputs.last_hidden_state[:,0,:].squeeze().numpy()

In [ ]:
import numpy as np
x=[get_embedding(t) for t in texts]
x=np.array(x)

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(x,labels,test_size=0.2,random_state=42)
clf=LogisticRegression(max_iter=1000)
clf.fit(xtrain,ytrain)
pred=clf.predict(xtest)

In [ ]:
print(classification_report(ytest,pred))

In [ ]:
for note in notes:
    emb=get_embedding(note).reshape(1,-1)
    pred=clf.predict(emb)[0]
    prob=clf.predict_proba(emb)[0]
    print(f"Prediction for {note}:","Adverse Event" if pred else "Normal")
    print(f"Confidence: {max(prob):.0%}")